# Experimento de 140 combinaciones, parte 6: estadística formal

¿Las diferencias entre modelos son reales o caben en la variación entre pliegues? Este cuaderno aplica la batería que pide la guía, con
el orden jerárquico que propone el documento de contexto (§5.8):

**Clasificación**
1. Friedman como prueba ómnibus (Friedman, 1937).
2. Nemenyi con diagrama de diferencia crítica, por niveles: los 7 modelos, los 4 balanceos y los 4 optimizadores (Demšar, 2006).
3. DeLong uno contra el resto en los finalistas, con Holm (DeLong et al., 1988; Sun y Xu, 2014; Holm, 1979).
4. Delta de Cliff (Cliff, 1993) e intervalos bootstrap BCa (Efron, 1987).

**Regresión**
5. Conjunto de confianza de modelos, MCS (Hansen et al., 2011).
6. Diebold-Mariano con la corrección HLN (Diebold y Mariano, 1995; Harvey et al., 1997), Clark-West para el par anidado (Clark y West, 2007)
   y Giacomini-White condicional (Giacomini y White, 2006).
7. d de Cohen.

Dos advertencias que cambian cómo se leen los resultados:

- **Con 5 pliegues, Nemenyi tiene poca potencia.** Con 7 modelos la diferencia crítica es de 4.03 rangos sobre un máximo de 6. Solo detecta
  diferencias enormes. Por eso se complementa con pruebas sobre las predicciones fuera de pliegue (DeLong, bootstrap), que usan las 58 mil plantas.
- **Las pruebas de series se aplican en orden espacial.** DM, GW y el bootstrap estacionario del MCS suponen un orden. Las pérdidas se ordenan
  por una curva de Hilbert sobre latitud y longitud, así la dependencia que corrigen es la autocorrelación espacial entre plantas vecinas.

## Pipeline

![Pipeline del experimento](fig_pipeline_experimento.png)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scikit_posthocs as sp
from sklearn.linear_model import Ridge
from sklearn.metrics import f1_score
from src import analisis, balancing, config, cv, models, stats

maestra, res = analisis.cargar()
b = analisis.base()
OPT = ["grilla", "aleatoria", "bayesiana", "genetica"]
orden_h = np.argsort(stats.indice_hilbert(b.tabla.latitude.values, b.tabla.longitude.values), kind="stable")
clf_m = maestra[maestra.tarea == "clf"]
mejores = analisis.mejores_por_modelo(res, "clf")
mejores = mejores.sort_values("puntaje_interno", ascending=False)
print("Modelos de clasificación, del mejor al peor puntaje interno:", list(mejores.index))

## 1. Friedman sobre todas las combinaciones de clasificación

Bloques = outer folds; tratamientos = combinaciones. Pregunta: ¿alguna combinación es distinta de las demás?

In [ ]:
por_pliegue = clf_m.pivot_table(index="pliegue", columns=["modelo", "balanceo", "optimizador"], values="f1_macro")
fr = stats.friedman_nemenyi(por_pliegue)
print(f"{por_pliegue.shape[1]} combinaciones, {por_pliegue.shape[0]} pliegues: chi² = {fr['chi2']:.1f}, p = {fr['p']:.2e}, "
      f"diferencia crítica de Nemenyi = {fr['cd']:.1f} rangos")

## 2. Diagramas de diferencia crítica por niveles

Las combinaciones unidas por una barra no se distinguen con Nemenyi al 5 %.

In [ ]:
def diagrama(tabla, titulo, archivo, ax):
    if tabla.shape[1] < 3:  # Friedman necesita al menos 3 tratamientos
        ax.axis("off"); ax.set_title(f"{titulo}: menos de 3 combinaciones completas, sin prueba"); return None
    r = stats.friedman_nemenyi(tabla)
    sp.critical_difference_diagram(r["rangos"], r["nemenyi"], ax=ax, label_fmt_left="{label} ({rank:.2f})  ",
                                   label_fmt_right="  ({rank:.2f}) {label}")
    ax.set_title(f"{titulo}\nFriedman p = {r['p']:.3g}, CD = {r['cd']:.2f}", fontsize=11)
    return r

# (a) cada modelo en su mejor combinación (elegida por puntaje interno)
t_mod = pd.DataFrame({m: clf_m[(clf_m.modelo == m) & (clf_m.balanceo == f.balanceo) & (clf_m.optimizador == f.optimizador)]
                      .set_index("pliegue").f1_macro for m, f in mejores.iterrows()})
mejor = mejores.index[0]
t_bal = clf_m[clf_m.modelo == mejor].groupby(["pliegue", "balanceo"]).f1_macro.mean().unstack()
t_opt = clf_m[clf_m.modelo == mejor].groupby(["pliegue", "optimizador"]).f1_macro.mean().unstack()

fig, axes = plt.subplots(3, 1, figsize=(10, 9))
r_mod = diagrama(t_mod, "Modelos (cada uno en su mejor combinación)", None, axes[0])
r_bal = diagrama(t_bal, f"Balanceos dentro de {mejor} (promedio de optimizadores)", None, axes[1])
r_opt = diagrama(t_opt, f"Optimizadores dentro de {mejor} (promedio de balanceos)", None, axes[2])
fig.tight_layout(); fig.savefig("fig_exp_cd.png", dpi=130, bbox_inches="tight"); plt.show()

In [ ]:
r_mod["nemenyi"].round(3) if r_mod else "sin prueba"

## 3. DeLong en los finalistas

Se comparan las curvas ROC de los tres modelos con mejor puntaje interno, con las predicciones fuera de pliegue de las 58 mil plantas. DeLong
es para problemas binarios; se aplica uno contra el resto por clase, empezando por la clase Baja, que es la que importa para no invertir mal.
Los p crudos se ajustan con Holm sobre todas las comparaciones de la tabla.

In [ ]:
finalistas = list(mejores.index[:3])
proba = {m: analisis.oof(mejores.loc[m])["proba"] for m in finalistas}
filas = []
for c, clase in enumerate(config.CLASES):
    yb = (b.y_clf == c).astype(int)
    for i, a in enumerate(finalistas):
        for z in finalistas[i + 1:]:
            r = stats.delong(yb, proba[a][:, c], proba[z][:, c])
            filas.append({"clase": clase, "comparación": f"{a} vs {z}", "AUC 1": r["auc1"], "AUC 2": r["auc2"], "z": r["z"], "p crudo": r["p"]})
dl = pd.DataFrame(filas)
dl["p Holm"] = stats.holm(dl["p crudo"])
dl.set_index(["clase", "comparación"]).round(4)

## 4. Tamaño del efecto e intervalos

- **Delta de Cliff** entre finalistas, sobre el F1 macro de los 5 pliegues: −1 a 1, con los umbrales de Romano et al. (2006).
- **Intervalo BCa del 95 %** del F1 macro fuera de pliegue. Se remuestrean bloques espaciales enteros, no plantas: las plantas de un
  mismo bloque no son independientes y remuestrearlas una por una daría intervalos demasiado angostos.

In [ ]:
filas = []
for i, a in enumerate(finalistas):
    for z in finalistas[i + 1:]:
        cd = stats.cliff_delta(t_mod[a], t_mod[z])
        filas.append({"comparación": f"{a} vs {z}", "delta de Cliff": cd["delta"], "magnitud": cd["magnitud"],
                      "d de Cohen (pareado)": stats.cohen_d_pareado(t_mod[a], t_mod[z])})
display(pd.DataFrame(filas).set_index("comparación").round(3))

filas = []
for m in finalistas:
    pred = analisis.oof(mejores.loc[m])["pred"].astype(int)
    f = lambda idx: f1_score(b.y_clf[idx], pred[idx], average="macro")
    lo, hi = stats.bca_por_bloques(f, b.grupos, n_boot=1000)
    filas.append({"modelo": m, "F1 macro fuera de pliegue": f(np.arange(len(pred))), "IC 95 % BCa (bloques)": f"[{lo:.3f}, {hi:.3f}]"})
pd.DataFrame(filas).set_index("modelo").round(4)

## 5. Regresión: conjunto de confianza de modelos

El MCS empieza con los 7 modelos (cada uno en su mejor combinación) y elimina uno por uno los que son peores con significancia, hasta que los que
quedan no se distinguen entre sí. Con nivel del 10 %, los que quedan son "los mejores" con 90 % de confianza. Pérdida = error al cuadrado, en orden
de Hilbert, con bootstrap estacionario (Politis y Romano, 1994).

In [ ]:
mej_r = analisis.mejores_por_modelo(res, "reg").sort_values("puntaje_interno", ascending=False)
y = b.y_reg
pred_r = {m: analisis.oof(mej_r.loc[m])["pred"] for m in mej_r.index}
perdidas = pd.DataFrame({m: ((y - p) ** 2)[orden_h] for m, p in pred_r.items()})
r_mcs = stats.mcs(perdidas, tamano=0.10, reps=500)
pd.DataFrame({"RMSE fuera de pliegue": np.sqrt(perdidas.mean()), "p del MCS": r_mcs["pvalores"].reindex(perdidas.columns).squeeze(),
              "en el conjunto (90 %)": [m in r_mcs["incluidos"] for m in perdidas.columns]}).round(4)

## 6. Comparaciones por pares en regresión

- **Diebold-Mariano con HLN**: el mejor modelo contra cada uno de los demás. Estadístico negativo = el mejor tiene menor pérdida.
- **Giacomini-White condicional**: pregunta si la diferencia de pérdida se puede predecir con la diferencia de la planta vecina anterior en la
  curva; capta si un modelo gana solo en ciertas zonas.
- **Clark-West**: para modelos anidados, donde DM está sesgado contra el modelo grande. El par natural es Ridge con las 15 variables frente a
  Ridge sin latitud ni longitud (el segundo está anidado en el primero). El Ridge sin coordenadas se ajusta aquí con los mismos pliegues
  externos y el mismo alpha.

In [ ]:
ref = mej_r.index[0]
filas = []
for m in mej_r.index[1:]:
    dm = stats.diebold_mariano(perdidas[ref], perdidas[m])
    gw = stats.giacomini_white(perdidas[ref].values, perdidas[m].values)
    filas.append({"comparación": f"{ref} vs {m}", "DM-HLN": dm["dm_hln"], "p DM": dm["p"], "GW": gw["gw"], "p GW": gw["p"]})
pares = pd.DataFrame(filas)
pares["p DM Holm"] = stats.holm(pares["p DM"])
pares.set_index("comparación").round(4)

In [ ]:
if "Ridge" in mej_r.index:
    import json
    alfa = json.loads(maestra[(maestra.tarea == "reg") & (maestra.modelo == "Ridge") & (maestra.optimizador == mej_r.loc["Ridge", "optimizador"])].params.iloc[0])["alpha"]
    sin = [config.VARIABLES.index(v) for v in config.VARIABLES if v not in ("latitude", "longitude")]
    p_sin = np.zeros(len(y))
    for tr, te in cv.externas(b.y_clf, b.grupos):
        m = balancing.construir(Ridge(alpha=alfa), "Ridge", "ninguno").fit(b.X[tr][:, sin], y[tr])
        p_sin[te] = m.predict(b.X[te][:, sin])
    cw = stats.clark_west(y[orden_h], p_sin[orden_h], pred_r["Ridge"][orden_h])
    print(f"Ridge sin lat/lon: RMSE {np.sqrt(np.mean((y - p_sin) ** 2)):.4f} | con lat/lon: {np.sqrt(np.mean((y - pred_r['Ridge']) ** 2)):.4f}")
    print(f"Clark-West = {cw['cw']:.2f}, p unilateral = {cw['p']:.2e} (H1: las coordenadas mejoran la predicción)")

## 7. d de Cohen en regresión

Sobre el RMSE de los 5 pliegues, el mejor modelo contra cada uno. Por convención, 0.2 es pequeño, 0.5 mediano y 0.8 grande (Cohen, 1988),
aunque con 5 pares el d es muy inestable.

In [ ]:
reg_m = maestra[maestra.tarea == "reg"]
rm = pd.DataFrame({m: reg_m[(reg_m.modelo == m) & (reg_m.optimizador == f.optimizador)].set_index("pliegue").rmse for m, f in mej_r.iterrows()})
pd.Series({f"{ref} vs {m}": stats.cohen_d_pareado(rm[m], rm[ref]) for m in rm.columns if m != ref}, name="d de Cohen (RMSE)").round(2).to_frame()

## 8. Interpretación

(se completa con la corrida completa)

## Referencias

- Clark, T. E. y West, K. D. (2007). Approximately normal tests for equal predictive accuracy in nested models. *Journal of Econometrics, 138*(1), 291-311. https://doi.org/10.1016/j.jeconom.2006.05.023
- Cliff, N. (1993). Dominance statistics: Ordinal analyses to answer ordinal questions. *Psychological Bulletin, 114*(3), 494-509. https://doi.org/10.1037/0033-2909.114.3.494
- Cohen, J. (1988). *Statistical power analysis for the behavioral sciences* (2.ª ed.). Lawrence Erlbaum.
- DeLong, E. R., DeLong, D. M. y Clarke-Pearson, D. L. (1988). Comparing the areas under two or more correlated receiver operating characteristic curves: A nonparametric approach. *Biometrics, 44*(3), 837-845. https://doi.org/10.2307/2531595
- Demšar, J. (2006). Statistical comparisons of classifiers over multiple data sets. *Journal of Machine Learning Research, 7*, 1-30.
- Diebold, F. X. y Mariano, R. S. (1995). Comparing predictive accuracy. *Journal of Business & Economic Statistics, 13*(3), 253-263. https://doi.org/10.1080/07350015.1995.10524599
- Efron, B. (1987). Better bootstrap confidence intervals. *Journal of the American Statistical Association, 82*(397), 171-185. https://doi.org/10.1080/01621459.1987.10478410
- Friedman, M. (1937). The use of ranks to avoid the assumption of normality implicit in the analysis of variance. *Journal of the American Statistical Association, 32*(200), 675-701. https://doi.org/10.1080/01621459.1937.10503522
- Giacomini, R. y White, H. (2006). Tests of conditional predictive ability. *Econometrica, 74*(6), 1545-1578. https://doi.org/10.1111/j.1468-0262.2006.00718.x
- Hansen, P. R., Lunde, A. y Nason, J. M. (2011). The model confidence set. *Econometrica, 79*(2), 453-497. https://doi.org/10.3982/ECTA5771
- Harvey, D., Leybourne, S. y Newbold, P. (1997). Testing the equality of prediction mean squared errors. *International Journal of Forecasting, 13*(2), 281-291. https://doi.org/10.1016/S0169-2070(96)00719-4
- Holm, S. (1979). A simple sequentially rejective multiple test procedure. *Scandinavian Journal of Statistics, 6*(2), 65-70.
- Politis, D. N. y Romano, J. P. (1994). The stationary bootstrap. *Journal of the American Statistical Association, 89*(428), 1303-1313. https://doi.org/10.1080/01621459.1994.10476870
- Romano, J., Kromrey, J. D., Coraggio, J. y Skowronek, J. (2006). Appropriate statistics for ordinal level data: Should we really be using t-test and Cohen's d for evaluating group differences on the NSSE and other surveys? Ponencia, Florida Association of Institutional Research.
- Sun, X. y Xu, W. (2014). Fast implementation of DeLong's algorithm for comparing the areas under correlated receiver operating characteristic curves. *IEEE Signal Processing Letters, 21*(11), 1389-1393. https://doi.org/10.1109/LSP.2014.2337313